In [1]:
from dataclasses import dataclass
from training import count_parameters, TrainConfig, train_waitk_student, TranslationDataset, save_and_log_checkpoint, load_training_checkpoint
import mlflow
import torch
import json
from mamba_ssm import Mamba2
from evaluation import SimulMTEvaluator, MTQualityScorer, WaitKMamba2Adapter
from model_classes import WaitKMamba2MT, SimulMamba2Config, WaitKTransformerMT, SimulTransformerConfig

In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("simulmt_waitk_mamba2_distillation")

<Experiment: artifact_location='/mlflow/artifacts/2', creation_time=1779451055489, experiment_id='2', last_update_time=1779451055489, lifecycle_stage='active', name='simulmt_waitk_mamba2_distillation', tags={}, trace_location=None, workspace='default'>

In [2]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

teacher_name = "./models/nllb_teacher"
tokenizer = AutoTokenizer.from_pretrained(teacher_name)

In [4]:
mamba2_config = SimulMamba2Config(
    vocab_size=len(tokenizer),

    d_model=512,
    num_layers=24,

    d_state=128,
    d_conv=4,
    expand=2,
    headdim=64,
    ngroups=1,

    dropout=0.1,

    max_source_len=64,
    max_target_len=64,

    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,

    tie_embeddings=True,
    separate_source_target_embeddings=True,
)

mamba2 = WaitKMamba2MT(mamba2_config)
count_parameters(mamba2)

Total parameters:     303,718,016
Trainable parameters: 303,718,016
Frozen parameters:    0


{'total': 303718016, 'trainable': 303718016, 'frozen': 0}

In [5]:
dataset = TranslationDataset("./data/train_dataset.hdf5")

In [8]:
train_cfg = TrainConfig(
    epochs=8,
    short_epochs=False,
    batches_per_epoch=2000,
    batch_size=192,
    gradient_accumulation_steps=8,

    wait_k=10,

    use_kl_loss=True,
    use_dataset_ce_loss=True,

    kl_weight=1.0,
    dataset_ce_weight=1.0,

    lr=3e-5,
    use_amp=True,
)

In [9]:
train_waitk_student(
    student=mamba2,
    train_dataset=dataset,
    model_cfg=mamba2_config,
    train_cfg=train_cfg,
    device="cuda",
    resume_from_checkpoint="./checkpoints/mamba_epoch_5.pt",
    checkpoint_name_prefix="mamba"
)

epoch 6/8:   0%|          | 0/14504 [00:00<?, ?it/s]

🏃 View run waitk-student at: http://localhost:5000/#/experiments/2/runs/16614573ca0241ef85bc0c9843abebf8
🧪 View experiment at: http://localhost:5000/#/experiments/2


KeyboardInterrupt: 

In [3]:
eval_dataset = TranslationDataset(
    "./data/val_dataset.hdf5",
    lazy=False,
)

In [ ]:
adapter = WaitKMamba2Adapter(
    model=mamba2,
    tokenizer=tokenizer,
    name="mamba2_wait_k",
    device="cuda",
    use_amp=True,
)

evaluator = SimulMTEvaluator(
    tokenizer=tokenizer,
    quality_scorer=MTQualityScorer(),
)

for k in [3, 5, 7, 9, 11, 13]:
    result = evaluator.evaluate(
        model=adapter,
        dataset=eval_dataset,
        wait_k=k,
        batch_size=1024,
        max_new_tokens=63
    )
    
    print(result.metrics)
    
    with open(f"./metrics/mamba_k{k}.json", "w") as file:
        json.dump(result.metrics, file)

In [4]:
from evaluation import WaitKTransformerAdapter

transformer = load_training_checkpoint("./models/transformer_wait10.pt", SimulTransformerConfig, WaitKTransformerMT)[0]

In [5]:
adapter = WaitKTransformerAdapter(
    model=transformer,
    tokenizer=tokenizer,
    name="transformer_wait_k",
    device="cuda",
    use_amp=True,
)

evaluator = SimulMTEvaluator(
    tokenizer=tokenizer,
    quality_scorer=MTQualityScorer(),
)

k = 10

result = evaluator.evaluate(
    model=adapter,
    dataset=eval_dataset,
    wait_k=k,
    batch_size=1024,
    max_new_tokens=63
)

print(result.metrics)

with open(f"./metrics/transformer_k{k}.json", "w") as file:
    json.dump(result.metrics, file)

Validating transformer_wait_k, wait_k=10:   0%|          | 0/303 [00:00<?, ?it/s]

{'BLEU': 29.489365020554242, 'chrF++': 52.25780140718085, 'TER': 61.685771415169846, 'AP': 0.8463807692628312, 'AL': 10.74056158963233, 'DAL': 11.482468849313264, 'LAAL': 9.132365275171225, 'ATD_text': 7.075138220135573, 'total_time_sec': 1063.1556936439993, 'ms_per_sentence': 3.436074120564944, 'target_tokens_per_sec': 8102.712567407433, 'source_tokens_per_sec': 7320.905156725095, 'first_token_latency_sec': 3.3621537817462466, 'peak_gpu_memory_mb': 2890.50537109375, 'generation_total_time_sec': 1016.5325524010077, 'generation_ms_per_sentence': 3.285390105041879, 'generation_target_tokens_per_sec': 8474.342488740807}
